# CS781 Capstone -- Phase 3: Rhetorical Role Prediction

## Stage 2: Shared setup and data

This notebook is built in stages, as agreed. This stage contains everything every one of the
8 experiments will reuse: the four Phase 1 backbones, the shared tokenizer, the Phase 3
data pipeline, the metrics / dev-calibration utilities, and the submission writer.

**No experiment-specific code (LoRA, LoRMA, IA3, Prompt/Prefix Tuning, KD, ensembling) is in
this stage.** Those come next, once you've confirmed this stage runs cleanly on your data.


## 1. Hugging Face Login

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()

from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF2")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


## 2. Check GPU

In [ ]:
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

CUDA available: True
GPU count: 1
GPU 0: Tesla T4


## 3. Install dependencies

`torchao` is uninstalled first as a known Kaggle/peft version conflict -- harmless if it was
never installed.

In [ ]:
!pip uninstall -y torchao -q 2>/dev/null
!pip -q install regex peft accelerate
import peft
print("peft version:", peft.__version__)

peft version: 0.20.0


## 4. Configuration

In [ ]:
import os, random, json, time, math, zipfile, shutil
import numpy as np
import torch

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILES = {
    "CL": "Phase_3_RR/Train/CL_train.json",
    "IT": "Phase_3_RR/Train/IT_train.json",
}
DEV_FILES = {
    "CL": "Phase_3_RR/Train/CL_dev.json",
    "IT": "Phase_3_RR/Train/IT_dev.json",
}
TEST_FILES = {
    "CL": "Phase_3_RR/Test/CL_test.json",
    "IT": "Phase_3_RR/Test/IT_test.json",
}
LABEL_VOCAB_FILE = "Phase_3_RR/Train/label_vocab.json"

# ---------------- Phase 1 checkpoints (all share the same tokenizer) ----------------
MODEL_PATHS = {
    "gpt2":    "/kaggle/input/datasets/himangshumahato/model1",
    "swiglu":  "/kaggle/input/datasets/himangshumahato/model2",
    "gqa":     "/kaggle/input/datasets/himangshumahato/model3",
    "prefixlm":"/kaggle/input/datasets/himangshumahato/model4",
}
MODEL_CONTEXT_LENGTH = 1024
N_EMBD, N_LAYER, N_HEAD = 768, 12, 12
N_KV_HEAD = 4   # GQA and Prefix-LM only

# MAX_SENT_LEN is set later, from the actual token-length distribution (Section 7 below)
MAX_SENT_LEN = None

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

BASE_OUTPUT_DIR = "/kaggle/working/cs781_phase3"
os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)

def experiment_dirs(name):
    """Every experiment gets its own folder: model, run_info.json, submission.zip."""
    d = os.path.join(BASE_OUTPUT_DIR, name)
    os.makedirs(d, exist_ok=True)
    return {
        "dir": d,
        "model_path": os.path.join(d, "best_model.pt"),
        "run_info_path": os.path.join(d, "run_info.json"),
        "prediction_path": os.path.join(d, "prediction.json"),
        "submission_zip": os.path.join(d, "submission.zip"),
        "loss_plot_path": os.path.join(d, "training_validation_loss.png"),
    }

print("Configuration loaded.")

Configuration loaded.


## 5. Tokenizer

Same from-scratch byte-level BPE used for all four Phase 1 models -- loaded once from the GQA checkpoint and reused everywhere, since it's the same file across all four model folders.

In [ ]:
%%writefile a1_tokenizer.py
"""From-scratch byte-level BPE tokenizer (GPT-2 style), no external tokenizer libs.

Differences from the plain-bytes version:
  * Text is first split into "words" with a GPT-2 style regex, and merges never
    cross word boundaries. Training and encoding then work on *unique words*
    (with counts) instead of the whole corpus, which is orders of magnitude faster.
  * Training uses incremental pair counts + a lazy max-heap (no full rescan per merge).
  * Encoding applies merges per word by rank, with a word -> ids cache.
  * One special token, <|endoftext|>, used as EOS / PAD / BOS.

Interface:
    tok.encode(text, add_eos=False) -> list[int]
    tok.decode(ids, skip_special_tokens=True) -> str
    tok.vocab_size / tok.get_vocab_size()
    tok.eos_token_id, tok.pad_token_id, tok.bos_token_id
    tok.save(path) / ByteLevelBPE.load(path)
"""

from __future__ import annotations

import heapq
import json
import time
from collections import Counter, defaultdict

import regex as re  # `pip install regex` (needed for \p{L} / \p{N})

# GPT-2 pre-tokenization pattern
_PAT = re.compile(
    r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
)

EOS_TOKEN = "<|endoftext|>"


class ByteLevelBPE:
    def __init__(self):
        # ids 0-255 = raw bytes; 256.. = learned merges; last id = <|endoftext|>
        self.merges: dict[tuple[int, int], int] = {}      # (a, b) -> new_id (id order == rank order)
        self.itos: dict[int, bytes] = {i: bytes([i]) for i in range(256)}
        self.eos_token_id = 256
        self.vocab_size = 257
        self._cache: dict[str, tuple] = {}

    # alias attributes so the tokenizer plugs into HF-style code
    @property
    def pad_token_id(self): return self.eos_token_id
    @property
    def bos_token_id(self): return self.eos_token_id

    def __len__(self): return self.vocab_size
    def get_vocab_size(self): return self.vocab_size

    def train(self, text: str, vocab_size: int, verbose: bool = False) -> None:
        """Learn (vocab_size - 257) merges; the final id is reserved for <|endoftext|>."""
        if vocab_size < 257:
            raise ValueError("vocab_size must be at least 257 (256 bytes + EOS)")
        num_merges = vocab_size - 257
        t0 = time.time()

        # 1) pre-tokenize and count unique words
        word_counts = Counter(_PAT.findall(text))
        words = [list(w.encode("utf-8")) for w in word_counts]
        freqs = list(word_counts.values())
        if verbose:
            print(f"[bpe] {len(words):,} unique words from {sum(freqs):,} total words")

        # 2) initial pair statistics
        stats: dict[tuple[int, int], int] = defaultdict(int)
        where: dict[tuple[int, int], set] = defaultdict(set)
        for wi, w in enumerate(words):
            f = freqs[wi]
            for p in zip(w, w[1:]):
                stats[p] += f
                where[p].add(wi)

        # lazy max-heap: (-count, a, b) -> highest count first, ties -> smallest (a, b)
        heap = [(-c, a, b) for (a, b), c in stats.items()]
        heapq.heapify(heap)

        self.merges, self.itos = {}, {i: bytes([i]) for i in range(256)}

        # 3) merge loop
        for i in range(num_merges):
            pair, count = None, 0
            while heap:
                negc, a, b = heapq.heappop(heap)
                if stats.get((a, b), 0) == -negc:   # skip stale heap entries
                    pair, count = (a, b), -negc
                    break
            if pair is None or count < 2:
                break

            new_id = 256 + i
            self.merges[pair] = new_id
            self.itos[new_id] = self.itos[pair[0]] + self.itos[pair[1]]

            touched = set()
            for wi in list(where[pair]):
                w, f = words[wi], freqs[wi]
                # remove this word's old pair contributions
                for p in zip(w, w[1:]):
                    stats[p] -= f
                    where[p].discard(wi)
                    touched.add(p)
                # apply merge (leftmost, non-overlapping)
                a, b = pair
                out, j, n = [], 0, len(w)
                while j < n:
                    if j < n - 1 and w[j] == a and w[j + 1] == b:
                        out.append(new_id); j += 2
                    else:
                        out.append(w[j]); j += 1
                words[wi] = out
                # add new pair contributions
                for p in zip(out, out[1:]):
                    stats[p] += f
                    where[p].add(wi)
                    touched.add(p)

            for p in touched:
                c = stats.get(p, 0)
                if c > 0:
                    heapq.heappush(heap, (-c, p[0], p[1]))
                else:
                    stats.pop(p, None)
                    where.pop(p, None)

            if verbose and (i % 1000 == 0 or i == num_merges - 1):
                print(f"[bpe] merge {i + 1}/{num_merges} | pair={pair} count={count:,} "
                      f"| elapsed={time.time() - t0:.1f}s")

        self.eos_token_id = 256 + len(self.merges)
        self.vocab_size = self.eos_token_id + 1
        self._cache = {}

    # Inference
    def _encode_word(self, word: str) -> tuple:
        hit = self._cache.get(word)
        if hit is not None:
            return hit
        ids = list(word.encode("utf-8"))
        merges = self.merges
        while len(ids) > 1:
            # pick the applicable merge learned earliest (lowest id)
            best, best_id = None, None
            for p in zip(ids, ids[1:]):
                nid = merges.get(p)
                if nid is not None and (best_id is None or nid < best_id):
                    best, best_id = p, nid
            if best is None:
                break
            a, b = best
            out, j, n = [], 0, len(ids)
            while j < n:
                if j < n - 1 and ids[j] == a and ids[j + 1] == b:
                    out.append(best_id); j += 2
                else:
                    out.append(ids[j]); j += 1
            ids = out
        res = tuple(ids)
        if len(self._cache) > 2_000_000:
            self._cache.clear()
        self._cache[word] = res
        return res

    def encode(self, text: str, add_eos: bool = False) -> list[int]:
        ids: list[int] = []
        for w in _PAT.findall(text):
            ids.extend(self._encode_word(w))
        if add_eos:
            ids.append(self.eos_token_id)
        return ids

    def decode(self, ids, skip_special_tokens: bool = True) -> str:
        if hasattr(ids, "tolist"):
            ids = ids.tolist()
        parts = []
        for i in ids:
            if i == self.eos_token_id:
                if not skip_special_tokens:
                    parts.append(EOS_TOKEN.encode("utf-8"))
                continue
            parts.append(self.itos[i])
        return b"".join(parts).decode("utf-8", errors="replace")

    def save(self, path: str) -> None:
        data = {
            "merges": [[a, b, nid] for (a, b), nid in self.merges.items()],
            "vocab_size": self.vocab_size,
            "eos_token_id": self.eos_token_id,
        }
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f)

    @classmethod
    def load(cls, path: str) -> "ByteLevelBPE":
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls()
        for a, b, nid in data["merges"]:
            tok.merges[(a, b)] = nid
            tok.itos[nid] = tok.itos[a] + tok.itos[b]
        tok.eos_token_id = data["eos_token_id"]
        tok.vocab_size = data["vocab_size"]
        return tok


def train_tokenizer(train_text: str, vocab_size: int):
    """Train a byte-level BPE tokenizer on `train_text` and return it."""
    tok = ByteLevelBPE()
    tok.train(train_text, vocab_size, verbose=True)
    return tok

Overwriting a1_tokenizer.py


In [ ]:
from a1_tokenizer import ByteLevelBPE, _PAT

tokenizer = ByteLevelBPE.load(os.path.join(MODEL_PATHS["gqa"], "bpe_tokenizer.json"))
print("Vocabulary size:", len(tokenizer))

# sanity: confirm all four checkpoints really do ship the identical tokenizer
for name, path in MODEL_PATHS.items():
    t = ByteLevelBPE.load(os.path.join(path, "bpe_tokenizer.json"))
    assert t.vocab_size == tokenizer.vocab_size, f"{name} tokenizer vocab size differs!"
print("All four checkpoints share the same tokenizer vocab size.")

Vocabulary size: 32000
All four checkpoints share the same tokenizer vocab size.


## 6. Load Phase 3 data

In [ ]:
from huggingface_hub import hf_hub_download

def hf_json(path):
    p = hf_hub_download(repo_id=HF_DATASET_REPO, filename=path, repo_type="dataset", token=HF_TOKEN)
    with open(p, encoding="utf-8") as f:
        return json.load(f)

train_docs, dev_docs, test_docs = {}, {}, {}
for domain, path in TRAIN_FILES.items():
    train_docs[domain] = hf_json(path)
for domain, path in DEV_FILES.items():
    dev_docs[domain] = hf_json(path)
for domain, path in TEST_FILES.items():
    test_docs[domain] = hf_json(path)

label_vocab = hf_json(LABEL_VOCAB_FILE)
print({d: len(v) for d, v in train_docs.items()}, "train docs")
print({d: len(v) for d, v in dev_docs.items()}, "dev docs")
print({d: len(v) for d, v in test_docs.items()}, "test docs")
print("label_vocab:", label_vocab)

{'CL': 40, 'IT': 40} train docs
{'CL': 5, 'IT': 5} dev docs
{'CL': 5, 'IT': 5} test docs
label_vocab: {'Fact': 0, 'Issue': 1, 'ArgumentPetitioner': 2, 'ArgumentRespondent': 3, 'Statute': 4, 'Dissent': 5, 'PrecedentReliedUpon': 6, 'PrecedentNotReliedUpon': 7, 'PrecedentOverruled': 8, 'RulingByLowerCourt': 9, 'RatioOfTheDecision': 10, 'RulingByPresentCourt': 11, 'None': 12}


## 7. Label vocabulary

13 labels, `"None"` included as a real string label, loaded from `label_vocab.json` rather than hardcoded, so a mismatch in spelling/casing surfaces immediately.

In [ ]:
# label_vocab.json's exact shape isn't guaranteed (list vs dict) -- handle both.
if isinstance(label_vocab, dict):
    LABELS = list(label_vocab.keys()) if all(isinstance(v, (int, float)) for v in label_vocab.values()) else list(label_vocab.values())
else:
    LABELS = list(label_vocab)

assert len(LABELS) == 13, f"expected 13 labels, got {len(LABELS)}: {LABELS}"
assert "None" in LABELS, "literal label 'None' missing from label_vocab"

LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}
print(LABELS)

['Fact', 'Issue', 'ArgumentPetitioner', 'ArgumentRespondent', 'Statute', 'Dissent', 'PrecedentReliedUpon', 'PrecedentNotReliedUpon', 'PrecedentOverruled', 'RulingByLowerCourt', 'RatioOfTheDecision', 'RulingByPresentCourt', 'None']


## 8. Flatten documents into sentence-level records

Keeps `doc_id`, `domain`, and `sent_idx` on every record so predictions can be reassembled in the exact original order per document later.

In [ ]:
def flatten(docs_by_domain, with_labels=True):
    records = []
    for domain, docs in docs_by_domain.items():
        if isinstance(docs, dict):
            items = list(docs.items())          # test: {doc_id: [sentences]}
        else:
            items = [(d["id"], d) for d in docs]  # train/dev: [{"id", "text", "labels"}, ...]

        for doc_id, doc in items:
            sents = doc if isinstance(doc, list) else doc["text"]
            labels = doc.get("labels") if (with_labels and isinstance(doc, dict)) else None
            if with_labels and labels is not None:
                assert len(sents) == len(labels), (doc_id, len(sents), len(labels))
            for i, s in enumerate(sents):
                rec = {"doc_id": doc_id, "domain": domain, "sent_idx": i, "text": s}
                if with_labels and labels is not None:
                    rec["label"] = labels[i]
                records.append(rec)
    return records

train_records = flatten(train_docs, with_labels=True)
dev_records   = flatten(dev_docs,   with_labels=True)
test_records  = flatten(test_docs,  with_labels=False)

print("train sentences:", len(train_records))
print("dev sentences:", len(dev_records))
print("test sentences:", len(test_records))

bad_labels = {r["label"] for r in train_records + dev_records} - set(LABELS)
assert not bad_labels, f"labels not in vocab: {bad_labels}"
print("all labels valid")

train sentences: 16845
dev sentences: 2142
test sentences: 2197
all labels valid


## 9. Sentence token-length distribution -> choose MAX_SENT_LEN

Data-driven, not assumed. Covers roughly the 99th percentile, rounded up to a clean multiple of 16.

In [ ]:
lengths = np.array([len(tokenizer.encode(r["text"])) for r in train_records])
percentiles = [50, 75, 90, 95, 99, 99.5, 100]
for p in percentiles:
    print(f"p{p:>5}: {np.percentile(lengths, p):.1f}")

p99 = np.percentile(lengths, 99)
MAX_SENT_LEN = int(math.ceil(p99 / 16) * 16)
MAX_SENT_LEN = min(MAX_SENT_LEN, MODEL_CONTEXT_LENGTH)
coverage = (lengths <= MAX_SENT_LEN).mean()
print(f"chosen MAX_SENT_LEN = {MAX_SENT_LEN}  (covers {coverage:.2%} of train sentences without truncation)")

p   50: 33.0
p   75: 51.0
p   90: 74.0
p   95: 93.0
p   99: 169.0
p 99.5: 208.0
p  100: 1196.0
chosen MAX_SENT_LEN = 176  (covers 99.16% of train sentences without truncation)


## 10. Dataset / collator -- sentence-only (experiments #1-#6)

Each sentence encoded independently, padded/truncated to `MAX_SENT_LEN`. The context-aware dataset for experiment #7 is a separate collator, built later.

In [ ]:
PAD_ID = tokenizer.eos_token_id

class SentenceDataset(torch.utils.data.Dataset):
    def __init__(self, records, with_labels=True):
        self.records = records
        self.with_labels = with_labels

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        ids = tokenizer.encode(r["text"])[:MAX_SENT_LEN]
        item = {"ids": ids, "idx": idx}
        if self.with_labels:
            item["label"] = LABEL2ID[r["label"]]
        return item


def sentence_collator(batch):
    L = MAX_SENT_LEN
    B = len(batch)
    input_ids = torch.full((B, L), PAD_ID, dtype=torch.long)
    attention_mask = torch.zeros((B, L), dtype=torch.bool)
    for i, item in enumerate(batch):
        n = len(item["ids"])
        input_ids[i, :n] = torch.tensor(item["ids"])
        attention_mask[i, :n] = True
    out = {"input_ids": input_ids, "attention_mask": attention_mask,
           "record_idx": torch.tensor([item["idx"] for item in batch])}
    if "label" in batch[0]:
        out["labels"] = torch.tensor([item["label"] for item in batch])
    return out

_test_batch = sentence_collator([SentenceDataset(train_records)[i] for i in range(4)])
print({k: v.shape for k, v in _test_batch.items()})

{'input_ids': torch.Size([4, 176]), 'attention_mask': torch.Size([4, 176]), 'record_idx': torch.Size([4]), 'labels': torch.Size([4])}


## 11. Class weights (joint CL+IT, inverse-frequency, clipped)

In [ ]:
from collections import Counter

train_label_counts = Counter(r["label"] for r in train_records)
total = sum(train_label_counts.values())
print(f"{'label':25s} {'count':>7s} {'%':>6s}")
for l in LABELS:
    c = train_label_counts.get(l, 0)
    print(f"{l:25s} {c:7d} {100*c/total:5.1f}%")

weights_list = []
for l in LABELS:
    c = train_label_counts.get(l, 0)
    weights_list.append(total / (len(LABELS) * c) if c > 0 else 0.0)
class_weights = torch.tensor(np.clip(weights_list, 0.1, 10.0), dtype=torch.float).to(device)
print("class weights:", class_weights.tolist())

label                       count      %
Fact                         4098  24.3%
Issue                         107   0.6%
ArgumentPetitioner           1613   9.6%
ArgumentRespondent           1348   8.0%
Statute                      1099   6.5%
Dissent                       284   1.7%
PrecedentReliedUpon          1945  11.5%
PrecedentNotReliedUpon        569   3.4%
PrecedentOverruled              3   0.0%
RulingByLowerCourt            874   5.2%
RatioOfTheDecision           4301  25.5%
RulingByPresentCourt          307   1.8%
None                          297   1.8%
class weights: [0.3161955177783966, 10.0, 0.8033287525177002, 0.9612531661987305, 1.1790438890457153, 4.562567710876465, 0.6662052869796753, 2.2772746086120605, 10.0, 1.4825735092163086, 0.30127161741256714, 4.2207465171813965, 4.36285924911499]


## 12. Backbone architectures (all four Phase 1 models)

In [ ]:
import math as _m
import torch.nn as nn
import torch.nn.functional as F

# ---------------- shared causal attention block (GPT-2 baseline) ----------------
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, n_positions, dropout=0.1):
        super().__init__()
        self.n_head = n_head; self.n_embd = n_embd
        self.head_dim = n_embd // n_head
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.attn_dropout = nn.Dropout(dropout); self.resid_dropout = nn.Dropout(dropout)
        self.register_buffer("bias", torch.tril(torch.ones(n_positions, n_positions)).view(1, 1, n_positions, n_positions))

    def forward(self, x, layer_past=None, attn_mask=None):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        if layer_past is not None:
            pk, pv = layer_past; k = torch.cat([pk, k], -2); v = torch.cat([pv, v], -2)
        present = (k, v)
        past_len = layer_past[0].size(-2) if layer_past is not None else 0
        seq_len = T + past_len
        att = (q @ k.transpose(-2, -1)) * (1.0 / _m.sqrt(self.head_dim))
        if attn_mask is not None:
            att = att.masked_fill(~attn_mask, float('-inf'))
        else:
            mask = self.bias[:, :, past_len:past_len + T, :seq_len] == 0
            att = att.masked_fill(mask, float('-inf'))
        att = self.attn_dropout(F.softmax(att, dim=-1))
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y)), present


class MLP(nn.Module):
    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd); self.gelu = nn.GELU(approximate='tanh')
        self.c_proj = nn.Linear(4 * n_embd, n_embd); self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


class TransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head, n_positions, dropout=0.1):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd); self.attn = CausalSelfAttention(n_embd, n_head, n_positions, dropout)
        self.ln_2 = nn.LayerNorm(n_embd); self.mlp = MLP(n_embd, dropout)

    def forward(self, x, layer_past=None, attn_mask=None):
        a, present = self.attn(self.ln_1(x), layer_past=layer_past, attn_mask=attn_mask)
        x = x + a; x = x + self.mlp(self.ln_2(x))
        return x, present


class GPT2ScratchLM(nn.Module):
    def __init__(self, vocab_size, n_positions, n_embd, n_layer, n_head, dropout=0.1):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, n_embd); self.wpe = nn.Embedding(n_positions, n_embd)
        self.drop = nn.Dropout(dropout)
        self.h = nn.ModuleList([TransformerBlock(n_embd, n_head, n_positions, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd); self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)


# ---------------- RMSNorm + SwiGLU (Model 2) ----------------
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps; self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight


class SwiGLUMLP(nn.Module):
    def __init__(self, n_embd, dropout=0.1, hidden_mult=4):
        super().__init__()
        hidden = int(n_embd * hidden_mult * 2 / 3)
        self.w1 = nn.Linear(n_embd, hidden, bias=False)   # gate
        self.w3 = nn.Linear(n_embd, hidden, bias=False)   # up
        self.w2 = nn.Linear(hidden, n_embd, bias=False)   # down
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.w2(F.silu(self.w1(x)) * self.w3(x)))


class CausalSelfAttentionRMS(CausalSelfAttention):
    pass   # identical mechanics; RMSNorm is applied outside, in the block


class RMSSwiGLUBlock(nn.Module):
    def __init__(self, n_embd, n_head, n_positions, dropout=0.1):
        super().__init__()
        self.ln_1 = RMSNorm(n_embd); self.attn = CausalSelfAttentionRMS(n_embd, n_head, n_positions, dropout)
        self.ln_2 = RMSNorm(n_embd); self.mlp = SwiGLUMLP(n_embd, dropout)

    def forward(self, x, layer_past=None, attn_mask=None):
        a, present = self.attn(self.ln_1(x), layer_past=layer_past, attn_mask=attn_mask)
        x = x + a; x = x + self.mlp(self.ln_2(x))
        return x, present


class RMSSwiGLULM(nn.Module):
    def __init__(self, vocab_size, n_positions, n_embd, n_layer, n_head, dropout=0.1):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, n_embd); self.wpe = nn.Embedding(n_positions, n_embd)
        self.drop = nn.Dropout(dropout)
        self.h = nn.ModuleList([RMSSwiGLUBlock(n_embd, n_head, n_positions, dropout) for _ in range(n_layer)])
        self.ln_f = RMSNorm(n_embd); self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)


# ---------------- Grouped-Query Attention (Model 3 / shared with Model 4) ----------------
class GroupedQueryAttention(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, n_positions, dropout=0.1):
        super().__init__()
        self.n_head = n_head; self.n_kv_head = n_kv_head
        self.head_dim = n_embd // n_head; self.num_groups = n_head // n_kv_head
        self.q_proj = nn.Linear(n_embd, n_embd)
        self.k_proj = nn.Linear(n_embd, n_kv_head * self.head_dim)
        self.v_proj = nn.Linear(n_embd, n_kv_head * self.head_dim)
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.attn_dropout = nn.Dropout(dropout); self.resid_dropout = nn.Dropout(dropout)
        self.register_buffer("bias", torch.tril(torch.ones(n_positions, n_positions)).view(1, 1, n_positions, n_positions))

    def forward(self, x, layer_past=None, attn_mask=None):
        B, T, C = x.size()
        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        if layer_past is not None:
            pk, pv = layer_past; k = torch.cat([pk, k], -2); v = torch.cat([pv, v], -2)
        present = (k, v)
        past_len = layer_past[0].size(-2) if layer_past is not None else 0
        seq_len = T + past_len
        k = k.repeat_interleave(self.num_groups, dim=1); v = v.repeat_interleave(self.num_groups, dim=1)
        att = (q @ k.transpose(-2, -1)) * (1.0 / _m.sqrt(self.head_dim))
        if attn_mask is not None:
            att = att.masked_fill(~attn_mask, float('-inf'))
        else:
            mask = self.bias[:, :, past_len:past_len + T, :seq_len] == 0
            att = att.masked_fill(mask, float('-inf'))
        att = self.attn_dropout(F.softmax(att, dim=-1))
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y)), present


class GQATransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, n_positions, dropout=0.1):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = GroupedQueryAttention(n_embd, n_head, n_kv_head, n_positions, dropout)
        self.ln_2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4*n_embd), nn.GELU(approximate='tanh'),
                                 nn.Linear(4*n_embd, n_embd), nn.Dropout(dropout))

    def forward(self, x, layer_past=None, attn_mask=None):
        a, present = self.attn(self.ln_1(x), layer_past=layer_past, attn_mask=attn_mask)
        x = x + a; x = x + self.mlp(self.ln_2(x))
        return x, present


class ScratchGroupedQueryLM(nn.Module):
    def __init__(self, vocab_size, n_positions, n_embd, n_layer, n_head, n_kv_head, dropout=0.1):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, n_embd); self.wpe = nn.Embedding(n_positions, n_embd)
        self.drop = nn.Dropout(dropout)
        self.h = nn.ModuleList([GQATransformerBlock(n_embd, n_head, n_kv_head, n_positions, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd); self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)


# ---------------- Prefix-LM (Model 4, same block type as GQA) ----------------
class PrefixLM(nn.Module):
    def __init__(self, vocab_size, n_positions, n_embd, n_layer, n_head, n_kv_head, dropout=0.1):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, n_embd); self.wpe = nn.Embedding(n_positions, n_embd)
        self.drop = nn.Dropout(dropout)
        self.h = nn.ModuleList([GQATransformerBlock(n_embd, n_head, n_kv_head, n_positions, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd); self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)

print("Backbone classes defined.")

Backbone classes defined.


## 13. Backbone registry and loader

In [ ]:
from safetensors.torch import load_file

BACKBONE_REGISTRY = {
    "gpt2":     {"cls": GPT2ScratchLM,       "kwargs": dict(n_embd=N_EMBD, n_layer=N_LAYER, n_head=N_HEAD)},
    "swiglu":   {"cls": RMSSwiGLULM,         "kwargs": dict(n_embd=N_EMBD, n_layer=N_LAYER, n_head=N_HEAD)},
    "gqa":      {"cls": ScratchGroupedQueryLM, "kwargs": dict(n_embd=N_EMBD, n_layer=N_LAYER, n_head=N_HEAD, n_kv_head=N_KV_HEAD)},
    "prefixlm": {"cls": PrefixLM,            "kwargs": dict(n_embd=N_EMBD, n_layer=N_LAYER, n_head=N_HEAD, n_kv_head=N_KV_HEAD)},
}

def load_backbone(name):
    """Fresh instance of the named Phase 1 backbone, weights loaded, lm_head dropped
    (not needed for classification)."""
    spec = BACKBONE_REGISTRY[name]
    model = spec["cls"](vocab_size=tokenizer.vocab_size, n_positions=MODEL_CONTEXT_LENGTH, **spec["kwargs"])
    state = load_file(os.path.join(MODEL_PATHS[name], "model.safetensors"))
    result = model.load_state_dict(state)
    print(f"[{name}] load_state_dict: {result}")
    if hasattr(model, "lm_head"):
        del model.lm_head
    return model

# smoke test: confirm every checkpoint actually loads cleanly before building anything on top
for name in BACKBONE_REGISTRY:
    m = load_backbone(name)
    n_params = sum(p.numel() for p in m.parameters())
    print(f"{name}: {n_params/1e6:.1f}M params\n")
    del m

[gpt2] load_state_dict: <All keys matched successfully>
gpt2: 110.4M params

[swiglu] load_state_dict: <All keys matched successfully>
swiglu: 110.4M params

[gqa] load_state_dict: <All keys matched successfully>
gqa: 101.0M params

[prefixlm] load_state_dict: <All keys matched successfully>
prefixlm: 101.0M params



## 14. Sentence classifier wrapper

Wraps any backbone with last-token pooling and a small MLP head. `bidirectional=True` is only
ever used for the separate Prefix-LM ablation; every other experiment leaves it `False`
(causal, matching Phase 1 pretraining).

In [ ]:
class SentenceClassifier(nn.Module):
    def __init__(self, lm, n_labels, n_embd, dropout=0.1, bidirectional=False):
        super().__init__()
        self.lm = lm
        self.bidirectional = bidirectional
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(n_embd, n_embd), nn.GELU(approximate='tanh'),
            nn.Dropout(dropout), nn.Linear(n_embd, n_labels)
        )

    def forward(self, input_ids, attention_mask=None, labels=None, class_weights=None, logit_offsets=None):
        B, T = input_ids.shape
        positions = torch.arange(T, device=input_ids.device)
        x = self.lm.wte(input_ids) + self.lm.wpe(positions)
        x = self.lm.drop(x)

        if self.bidirectional:
            attn_mask = torch.ones(B, 1, T, T, dtype=torch.bool, device=input_ids.device)
            if attention_mask is not None:
                attn_mask = attn_mask & attention_mask[:, None, None, :].bool()
        else:
            attn_mask = None   # falls back to each block's own causal buffer
            if attention_mask is not None:
                causal = torch.tril(torch.ones(T, T, dtype=torch.bool, device=input_ids.device)).view(1, 1, T, T)
                attn_mask = causal & attention_mask[:, None, None, :].bool()

        for block in self.lm.h:
            x, _ = block(x, attn_mask=attn_mask)
        x = self.lm.ln_f(x)

        # last real (non-pad) token per sequence
        if attention_mask is not None:
            last_idx = attention_mask.sum(dim=1) - 1
        else:
            last_idx = torch.full((B,), T - 1, device=input_ids.device)
        pooled = x[torch.arange(B, device=input_ids.device), last_idx]

        logits = self.head(self.dropout(pooled))
        if logit_offsets is not None:
            logits = logits + logit_offsets

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels, weight=class_weights)
        return {"loss": loss, "logits": logits}

print("SentenceClassifier defined.")

SentenceClassifier defined.


## 15. Metrics and dev-only logit-offset calibration

`macro_prf` computes macro-F1/P/R, with an optional domain filter for the CL/IT split the brief
requires. `calibrate_logit_offsets` does a small coordinate-ascent search over per-class additive
offsets to raise dev macro-F1 -- fit once on dev, then frozen and reused for test.

In [ ]:
def macro_prf(y_true, y_pred, n_labels=len(LABELS)):
    cnts = np.zeros((n_labels, 3))   # tp, pred, gold
    for t, p in zip(y_true, y_pred):
        cnts[p, 1] += 1
        cnts[t, 2] += 1
        if t == p:
            cnts[t, 0] += 1
    P = np.divide(cnts[:, 0], cnts[:, 1], out=np.zeros(n_labels), where=cnts[:, 1] > 0)
    R = np.divide(cnts[:, 0], cnts[:, 2], out=np.zeros(n_labels), where=cnts[:, 2] > 0)
    F = np.divide(2 * P * R, P + R, out=np.zeros(n_labels), where=(P + R) > 0)
    return P, R, F


def print_scores(y_true, y_pred, domains=None, title=""):
    print(f"--- {title} ---")
    P, R, F = macro_prf(y_true, y_pred)
    print(f"{'label':25s} {'P':>6s} {'R':>6s} {'F1':>6s}")
    for i, l in enumerate(LABELS):
        print(f"{l:25s} {P[i]:6.3f} {R[i]:6.3f} {F[i]:6.3f}")
    print(f"MACRO  P {P.mean():.4f}  R {R.mean():.4f}  F1 {F.mean():.4f}")
    if domains is not None:
        for dom in sorted(set(domains)):
            idx = [i for i, d in enumerate(domains) if d == dom]
            dP, dR, dF = macro_prf([y_true[i] for i in idx], [y_pred[i] for i in idx])
            print(f"[{dom}]  P {dP.mean():.4f}  R {dR.mean():.4f}  F1 {dF.mean():.4f}")
    return F.mean()


@torch.no_grad()
def get_logits(model, loader):
    model.eval()
    all_logits, all_labels = [], []
    for b in loader:
        b = {k: v.to(device) for k, v in b.items()}
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            out = model(input_ids=b["input_ids"], attention_mask=b["attention_mask"])
        all_logits.append(out["logits"].float().cpu())
        if "labels" in b:
            all_labels.append(b["labels"].cpu())
    logits = torch.cat(all_logits)
    labels = torch.cat(all_labels) if all_labels else None
    return logits, labels


def calibrate_logit_offsets(logits, labels, n_labels=len(LABELS), n_rounds=3, step=0.5, n_steps=9):
    """Coordinate ascent: for each label in turn, try a small range of additive offsets and
    keep whichever improves dev macro-F1. Fit on dev only; the returned offsets are then frozen
    and applied identically at test time."""
    offsets = torch.zeros(n_labels)
    y_true = labels.numpy()
    candidates = np.linspace(-step * n_steps, step * n_steps, 2 * n_steps + 1)
    best_f1 = macro_prf(y_true, (logits + offsets).argmax(-1).numpy())[2].mean()
    for _ in range(n_rounds):
        improved = False
        for c in range(n_labels):
            base = offsets[c].item()
            best_c, best_c_f1 = base, best_f1
            for delta in candidates:
                offsets[c] = delta
                f1 = macro_prf(y_true, (logits + offsets).argmax(-1).numpy())[2].mean()
                if f1 > best_c_f1:
                    best_c, best_c_f1 = delta, f1
            offsets[c] = best_c
            if best_c_f1 > best_f1 + 1e-6:
                best_f1 = best_c_f1
                improved = True
        if not improved:
            break
    print(f"calibration: dev macro-F1 {best_f1:.4f} (offsets: {[round(o,2) for o in offsets.tolist()]})")
    return offsets, best_f1

print("Metrics utilities defined.")

Metrics utilities defined.


## 16. Submission writer

In [ ]:
def write_submission(paths, model, bidirectional_override=None):
    """Runs the model over test_records, reassembles per-document predictions in original
    sentence order, writes prediction.json and submission.zip."""
    test_ds = SentenceDataset(test_records, with_labels=False)
    test_loader = torch.utils.data.DataLoader(test_ds, batch_size=32, shuffle=False, collate_fn=sentence_collator)

    model.eval()
    all_logits = []
    with torch.no_grad():
        for b in test_loader:
            b = {k: v.to(device) for k, v in b.items() if k != "record_idx"}
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                out = model(input_ids=b["input_ids"], attention_mask=b["attention_mask"])
            all_logits.append(out["logits"].float().cpu())
    logits = torch.cat(all_logits)
    preds = logits.argmax(-1).tolist()

    by_doc = {}
    for rec, p in zip(test_records, preds):
        by_doc.setdefault(rec["doc_id"], {})[rec["sent_idx"]] = ID2LABEL[p]

    prediction = {}
    for domain, docs in test_docs.items():
        items = docs.items() if isinstance(docs, dict) else [(d["id"], d) for d in docs]
        for doc_id, doc in items:
            sents = doc if isinstance(doc, list) else doc["text"]
            n = len(sents)
            ordered = [by_doc[doc_id][i] for i in range(n)]
            assert len(ordered) == n, (doc_id, len(ordered), n)
            prediction[doc_id] = ordered

    assert set(prediction) == {doc["id"] for docs in test_docs.values() for doc in docs}

    with open(paths["prediction_path"], "w", encoding="utf-8") as f:
        json.dump(prediction, f, ensure_ascii=False)
    with zipfile.ZipFile(paths["submission_zip"], "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(paths["prediction_path"], arcname="prediction.json")

    print("saved:", paths["submission_zip"])
    with zipfile.ZipFile(paths["submission_zip"]) as z:
        print(z.namelist())
    return prediction

print("Submission writer defined.")

Submission writer defined.


## 17. Generic training loop

Shared by every experiment; what differs per experiment is which parameters are passed to the optimizer (full fine-tune, frozen+adapter, etc.), supplied from outside.

In [ ]:
def train_classifier(model, optimizer, scheduler, train_loader, dev_loader, dev_domains,
                     class_weights, epochs, patience, paths, scaler=None):
    scaler = scaler or torch.amp.GradScaler("cuda", enabled=USE_AMP)
    history = {"train": [], "dev_f1": []}
    best_f1, bad_epochs = 0.0, 0
    best_offsets = torch.zeros(len(LABELS))
    t0 = time.time()

    for epoch in range(epochs):
        model.train()
        run, cnt = 0.0, 0
        optimizer.zero_grad()
        for i, b in enumerate(train_loader):
            b = {k: v.to(device) for k, v in b.items() if k != "record_idx"}
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                out = model(**b, class_weights=class_weights)
            scaler.scale(out["loss"]).backward()
            run += out["loss"].item(); cnt += 1
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad()
            if scheduler is not None:
                scheduler.step()

        logits, labels = get_logits(model, dev_loader)
        offsets, f1 = calibrate_logit_offsets(logits, labels)
        history["train"].append(run / cnt); history["dev_f1"].append(f1)
        print(f"epoch {epoch+1}/{epochs} | train loss {run/cnt:.4f} | dev macro-F1 (calibrated) {f1:.4f} | {time.time()-t0:.0f}s")

        if f1 > best_f1:
            best_f1, bad_epochs, best_offsets = f1, 0, offsets
            torch.save(model.state_dict(), paths["model_path"])
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"early stopping at epoch {epoch+1} (best dev macro-F1 {best_f1:.4f})")
                break

    model.load_state_dict(torch.load(paths["model_path"], weights_only=True))
    print("restored best checkpoint, dev macro-F1:", best_f1)
    return model, history, best_offsets, best_f1

print("Training loop defined.")

Training loop defined.


---

## Stage 2 complete

Everything above is shared infrastructure: four backbones load and smoke-test cleanly, the
tokenizer and data pipeline are built, `MAX_SENT_LEN` is set from the real distribution (not
assumed), and metrics/calibration/submission utilities are ready.

